# Exploratory Data Analysis (EDA)
**Project:** Skin Cancer Detection System v3

Notebook ini bertujuan untuk menganalisis distribusi kelas, resolusi, dan pola warna dari subset dataset ISIC yang terdiri atas 9 kelas lesi kulit.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

## 1. Setup Data Paths
Mengatur path ke direktori training dan mendata semua subdirektori (kelas).

In [ ]:
base_dir = Path('../data')
train_dir = base_dir / 'Train'

if not train_dir.exists():
    print(f"Direktori tidak ditemukan: {train_dir}")
else:
    classes = [d.name for d in train_dir.iterdir() if d.is_dir()]
    print(f"Ditemukan {len(classes)} kelas:")
    for c in classes:
        print(f" - {c}")

## 2. Analisis Distribusi Kelas
Mengevaluasi imbalanced data pada dataset latih.

In [ ]:
class_counts = {}
if train_dir.exists():
    for c in classes:
        files = list((train_dir / c).glob('*.jpg')) + list((train_dir / c).glob('*.png'))
        class_counts[c] = len(files)
        
    df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Count'])
    df_counts = df_counts.sort_values(by='Count', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_counts, x='Count', y='Class')
    plt.title('Distribusi Data per Kelas (Train Dataset)')
    plt.xlabel('Jumlah Gambar')
    plt.ylabel('Kelas')
    plt.tight_layout()
    plt.show()

## 3. Visualisasi Sample Gambar
Menampilkan satu contoh variasi visual untuk mewakili setiap kelas.

In [ ]:
if train_dir.exists():
    plt.figure(figsize=(15, 10))
    for i, row in enumerate(df_counts.itertuples()):
        c = row.Class
        files = list((train_dir / c).glob('*.*'))
        if files:
            img_path = str(files[0])
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.subplot(3, 3, i + 1)
                plt.imshow(img)
                plt.title(c)
                plt.axis('off')
    plt.suptitle('Sample Lesi Kulit per Kelas', fontsize=16)
    plt.tight_layout()
    plt.show()

## 4. Sampling Analisis Dimensi & Resolusi
Mengekstrak bentuk file secara geometris dari sebagian dataset (100 gambar per kelas) untuk mengecek standar resolusinya.

In [ ]:
dimensions = []
if train_dir.exists():
    for c in classes:
        files = list((train_dir / c).glob('*.*'))[:100] # Ambil max 100 sample saja
        for f in files:
            img = cv2.imread(str(f))
            if img is not None:
                h, w, _ = img.shape
                dimensions.append({'Class': c, 'Width': w, 'Height': h})
                
    df_dims = pd.DataFrame(dimensions)
    
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_dims, x='Width', y='Height', alpha=0.5)
    plt.title('Sebaran Resolusi Gambar')
    plt.show()

## 5. Analisis Color Histogram (Kanal R, G, B)
Menentukan corak warna dominan dari sebuah random sample melanoma untuk optimasi augmentasi.

In [ ]:
if train_dir.exists():
    melanoma_files = list((train_dir / 'melanoma').glob('*.*'))
    if melanoma_files:
        img_path = str(melanoma_files[0])
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            colors = ('r', 'g', 'b')
            
            plt.figure(figsize=(10, 4))
            
            plt.subplot(1, 2, 1)
            plt.imshow(img)
            plt.axis('off')
            plt.title('Sumber Gambar (Melanoma)')
            
            plt.subplot(1, 2, 2)
            for i, c in enumerate(colors):
                hist = cv2.calcHist([img], [i], None, [256], [0, 256])
                plt.plot(hist, color=c)
            plt.title('Color Histogram')
            plt.xlabel('Bins')
            plt.ylabel('Frekuensi')
            plt.tight_layout()
            plt.show()